# データの可視化

## リスクウェイトとトランザクションデータとの相関分析

In [0]:
!pip install japanize-matplotlib

In [0]:
%restart_python

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import japanize_matplotlib
from pyspark.sql.functions import *
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    IntegerType,
    LongType,
    TimestampType,
)

from datetime import date, timedelta
import random


plt.rcParams['axes.unicode_minus'] = False

In [0]:
account_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("open_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("account_type", StringType(), False),
    StructField("risk_weights", IntegerType(), False),
])

regions = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州・沖縄"]
account_types = ["普通", "当座", "定期"]
risk_categories = ["低", "中", "高"]
risk_weights = [10, 5, 1]

today = date.today()
SCHEMA = "workspace.datasets"

NUM_ACCOUNTS = 10_000

accounts = [
    {
        "account_id": f"ACC{idx + 1:05}",
        "open_date": today - timedelta(days=random.randint(0, 365 * 10)),
        "region": random.choice(regions),
        "account_type": random.choice(account_types),
        "risk_weights": random.choice(risk_weights),
    }
    for idx in range(NUM_ACCOUNTS)
]

df_account = spark.createDataFrame(accounts, schema=account_schema)
df_account.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.account")

In [0]:
# データ読み込み
df_account_fraud = spark.read.table("workspace.datasets.account").select("account_id", "risk_weights")
df_transaction = spark.read.table("workspace.datasets.transaction")


In [0]:
fraud_sdf = (
    df_account_fraud.join(df_transaction, on="account_id", how="left")
)
display(fraud_sdf)

In [0]:
numeric_cols = [
    c for c, t in fraud_sdf.dtypes
    if t in ("int", "bigint", "float", "double")
]

corr_list = [
    (col, fraud_sdf.stat.corr("risk_weights", col))
    for col in numeric_cols
    if col != "risk_weights"
]

display(
    spark.createDataFrame(corr_list, ["column", "correlation"])
         .orderBy(desc("correlation"))
)